# Graph round-trip: DB ↔ NetworkX

Demonstrates:
1. Exporting a block group to a NetworkX `DiGraph` with `bg.to_networkx()`
2. Inspecting node and edge attributes (including the `sequence` field)
3. **How `node_id` controls deduplication** — reuse it to reference an existing
   node; omit it to create a fresh one even for identical sequences
4. A clean round-trip: export → re-import unchanged → verify node IDs match
5. A modification round-trip: build a new graph with changed sequences → re-import

`GenGraph` is a typed `nx.DiGraph` subclass that enforces `Block`-typed nodes.
Use it when building graphs from scratch; `bg.to_networkx()` returns a plain
`DiGraph` whose node keys are already `Block` objects.

In [1]:
import tempfile, gen, networkx as nx
from gen import GenGraph, Block

tmp = tempfile.TemporaryDirectory()
repo = gen.Repository(tmp.name + '/.gen')

## 1. Create an initial block group from a sequence

In [2]:
bg = repo.create_block_group_from_sequence(
    name='example',
    sequence='ACGTACGT',
)
print(bg)

BlockGroup(def7344d58a6459431b61bf9013e565752d10921e67e813ce388f7a45b483a3f, default, reference, example)


## 2. Export to NetworkX and inspect the graph

Each node key is a `Block` object.  The node *attributes* carry everything
needed for a faithful round-trip:

| Attribute | Meaning |
|---|---|
| `node_id` | 64-char hex string — the node’s identity in the DB |
| `sequence` | **full** underlying sequence string |
| `sequence_start` | first exposed byte (0-based, inclusive) |
| `sequence_end` | last exposed byte (0-based, exclusive) |

Edge attributes include `source_strand`, `target_strand`, and `weights`
(one entry per chromosome/phasing variant).

`to_networkx()` includes `StartBlock`/`EndBlock` sentinel boundary nodes by
default.  Pass `include_sentinels=False` to get content nodes only.

In [3]:
G = bg.to_networkx()   # includes StartBlock/EndBlock sentinel boundary nodes
print(f'Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}')
print('(2 nodes are boundary sentinels with no attrs)\n')

for node, attrs in G.nodes(data=True):
    if 'node_id' not in attrs:   # sentinel boundary node — no attrs
        print('Sentinel:', node)
        continue
    print('Node:', node)
    print('  node_id       :', attrs['node_id'])
    print('  sequence      :', attrs['sequence'])
    print('  sequence_start:', attrs['sequence_start'])
    print('  sequence_end  :', attrs['sequence_end'])
    window = attrs['sequence'][attrs['sequence_start']:attrs['sequence_end']]
    print('  window        :', window)

for src, dst, attrs in G.edges(data=True):
    print(f'\nEdge {src} → {dst}')
    print('  source_strand:', attrs.get('source_strand'))
    print('  target_strand:', attrs.get('target_strand'))

Nodes: 3  Edges: 2
(2 nodes are boundary sentinels with no attrs)

Node: Block("ACGTACGT", 0..8)
  node_id       : 019dccb7868f71f1ac42eaabc0c6542100000000000000000000000000000000
  sequence      : ACGTACGT
  sequence_start: 0
  sequence_end  : 8
  window        : ACGTACGT
Sentinel: Block("", 0..0)
Sentinel: Block("", 0..0)

Edge Block("ACGTACGT", 0..8) → Block("", 0..0)
  source_strand: +
  target_strand: +

Edge Block("", 0..0) → Block("ACGTACGT", 0..8)
  source_strand: +
  target_strand: +


## 3. The role of `node_id` in deduplication

In gen, a **node** is identified by its `node_id` (a UUID-7 hash).  Multiple
block groups can share nodes — they just reference the same `node_id`.

**Sequence** content is separately deduplicated by a content hash.  So two
nodes can carry the same DNA string while still being distinct graph nodes.

When you call `create_block_group_from_graph`:
- **`node_id` present** → the existing node is reused (INSERT is silently
  ignored on collision); the new block group references the same DB row.
- **`node_id` absent** → a brand-new UUID-7 is generated; you get a fresh
  node even if the sequence is identical to an existing one.

In [4]:
# --- with node_id preserved: re-import references the same DB nodes ---
G_with_ids = bg.to_networkx()   # node_id is carried on each Block key
bg_reuse = repo.create_block_group_from_graph(G_with_ids, name='reuse')

orig_ids  = {a['node_id'] for _, a in bg.to_networkx(include_sentinels=False).nodes(data=True)}
reuse_ids = {a['node_id'] for _, a in bg_reuse.to_networkx(include_sentinels=False).nodes(data=True)}
print('node_id sets match (shared nodes):', orig_ids == reuse_ids)

# --- with fresh Block objects (no node_id) — brand-new UUID-7s ---
G_fresh = GenGraph()
for node in bg.to_networkx(include_sentinels=False).nodes():
    G_fresh.add_node(Block(node._node_sequence))  # omit node_id → new UUID-7

bg_fresh = repo.create_block_group_from_graph(G_fresh, name='fresh')
fresh_ids = {a['node_id'] for _, a in bg_fresh.to_networkx(include_sentinels=False).nodes(data=True)}
print('node_id overlap when fresh blocks used:', orig_ids & fresh_ids)
print('(empty set = all-new nodes, as expected)')

node_id sets match (shared nodes): True
node_id overlap when fresh blocks used: set()
(empty set = all-new nodes, as expected)


## 4. Clean round-trip

Export → re-import unchanged.  The re-imported block group is independent
(different `BlockGroup.id`) but references the same underlying nodes.

In [5]:
G_export = bg.to_networkx()
bg_reimport = repo.create_block_group_from_graph(G_export, name='reimport')

d_orig     = bg.to_dict()
d_reimport = bg_reimport.to_dict()

ids_orig     = {v['node_id'] for v in d_orig['nodes'].values()}
ids_reimport = {v['node_id'] for v in d_reimport['nodes'].values()}

print('Block group IDs identical:', bg.id == bg_reimport.id, '(expected False)')
print('Node IDs identical        :', ids_orig == ids_reimport, '(expected True)')

Block group IDs identical: False (expected False)
Node IDs identical        : True (expected True)


## 5. Modification round-trip

Export → build a new graph with changed sequences → re-import.

Two strategies:

**A. Keep `node_id`** — pass the original `node_id` to the new `Block`.
Gen reuses the existing node row (INSERT OR IGNORE on `node_id`), so the
stored sequence is unchanged.  Use this when you want the new block group
to reference the *same* graph nodes as the original.

**B. Omit `node_id`** — a brand-new UUID-7 node is created.  The old node
still exists in the DB.  Use this when you want a derived graph that is
independent of the original.

In [6]:
# Strategy A: pass original node_id — existing DB node is reused (INSERT OR IGNORE).
# The stored sequence is unchanged; node_ids will match the original.
G_mod_a = GenGraph()
for node in bg.to_networkx(include_sentinels=False).nodes():
    G_mod_a.add_node(Block(node.sequence, node_id=node.node_id))

bg_mod_a = repo.create_block_group_from_graph(G_mod_a, name='mutated_keep_id')
d_mod_a  = bg_mod_a.to_dict()

ids_mod_a  = {v['node_id']  for v in d_mod_a['nodes'].values()}
seqs_mod_a = {v['sequence'] for v in d_mod_a['nodes'].values()}
print('Strategy A — node_ids same as original:', ids_mod_a == ids_orig)
print('Strategy A — sequences               :', seqs_mod_a)

Strategy A — node_ids same as original: True
Strategy A — sequences               : {'ACGTACGT'}


In [7]:
# Strategy B: omit node_id — brand-new nodes, independent of the original.
G_mod_b = GenGraph()
for node in bg.to_networkx(include_sentinels=False).nodes():
    G_mod_b.add_node(Block('GGGGGGGG'))  # no node_id → fresh UUID-7

bg_mod_b = repo.create_block_group_from_graph(G_mod_b, name='mutated_new_id')
d_mod_b  = bg_mod_b.to_dict()

ids_mod_b  = {v['node_id']  for v in d_mod_b['nodes'].values()}
seqs_mod_b = {v['sequence'] for v in d_mod_b['nodes'].values()}
print('Strategy B — node_ids same as original:', ids_mod_b == ids_orig)
print('Strategy B — sequences               :', seqs_mod_b)
print('Strategy B — node_ids overlap with A  :', ids_mod_a & ids_mod_b)

Strategy B — node_ids same as original: False
Strategy B — sequences               : {'GGGGGGGG'}
Strategy B — node_ids overlap with A  : set()


## 6. Sequence window (non-zero `sequence_start`)

When a block group is created with `sequence_start`/`sequence_end`, the node
stores the **full** sequence in the DB but only exposes a window in the graph.
`to_networkx()` exports the full sequence so the window indices remain valid.

In [8]:
bg_win = repo.create_block_group_from_sequence(
    name='windowed',
    sequence='ACGTACGT',
    sequence_start=2,
    sequence_end=6,
)
G_win = bg_win.to_networkx(include_sentinels=False)

for node, attrs in G_win.nodes(data=True):
    s, e = attrs['sequence_start'], attrs['sequence_end']
    print('full sequence  :', attrs['sequence'])
    print('window [%d:%d]  :' % (s, e), attrs['sequence'][s:e])

# Round-trip the windowed block group.
bg_win_rt = repo.create_block_group_from_graph(G_win, name='windowed_rt')
G_win_rt  = bg_win_rt.to_networkx(include_sentinels=False)

for node, attrs in G_win_rt.nodes(data=True):
    s, e = attrs['sequence_start'], attrs['sequence_end']
    print('round-trip window [%d:%d]:', attrs['sequence'][s:e])

full sequence  : ACGTACGT
window [2:6]  : GTAC
round-trip window [%d:%d]: GTAC


## 7. Multi-node graph round-trip

Build a two-node linear graph, export, verify, re-import.

In [9]:
G_multi = GenGraph()
nodeA = Block('AAAA')
nodeB = Block('CCCC')
G_multi.add_edge(nodeA, nodeB)

bg_multi = repo.create_block_group_from_graph(
    G_multi, name='multi', path_name='linear'
)
print('Created multi-node bg:', bg_multi)

# Export and inspect content nodes.
G_multi_export = bg_multi.to_networkx(include_sentinels=False)
print(f'Exported content nodes: {G_multi_export.number_of_nodes()}')
for node, attrs in G_multi_export.nodes(data=True):
    print(f'  {node}  seq={attrs["sequence"]}  id={attrs["node_id"][:8]}...')

# Re-import: Block keys carry node_ids, so nodes are reused.
bg_multi_rt = repo.create_block_group_from_graph(G_multi_export, name='multi_rt')
ids_src = {a['node_id'] for _, a in G_multi_export.nodes(data=True)}
ids_rt  = {a['node_id'] for _, a in bg_multi_rt.to_networkx(include_sentinels=False).nodes(data=True)}
print('Node IDs preserved in round-trip:', ids_src == ids_rt)

Created multi-node bg: BlockGroup(3ff73e96f4149e26432c007c2b38de375119bfe8a2858e4abfd18f107c25692e, default, reference, multi)
Exported content nodes: 2
  Block("AAAA", 0..4)  seq=AAAA  id=019dccb7...
  Block("CCCC", 0..4)  seq=CCCC  id=019dccb7...
Node IDs preserved in round-trip: True
